# PANGAEA batch metadata extraction (iNaturalist standard)

Loop over all CSV files under `data/experiment/pangea` (or the `pangaea_datasets` fallback), run the full pipeline with the `croissant_inaturalist_standard`, and save JSON outputs under `outputs/pangaea_inat_<model_slug>/`.

This notebook is configured for `RedHatAI/gemma-4-31B-it-NVFP4` on the SURF provider.

In [1]:
import json
import logging
import os
import re
import sys
from pathlib import Path

BASE = Path('../..').resolve()
sys.path.insert(0, str(BASE))

from dotenv import load_dotenv

# Load credentials/defaults, then force this notebook run to one model/provider.
load_dotenv(BASE / '.env')
os.environ['LLM_PROVIDER'] = 'surf'
os.environ['LLM_MODEL'] = 'default-text-large'

from src.config import LLM_PROVIDER, get_model_name
from src.orchestrator import run_metadata_extraction
from src.standards import METADATA_STANDARDS

STANDARD_NAME = 'croissant_inaturalist_standard'
TOPOLOGY = 'single'

LLM_NAME = get_model_name()
LLM_SLUG = re.sub(r'[^\w.\-]+', '_', LLM_NAME.replace('/', '_'))
EXPERIMENT_RUN = f'pangaea_inat_{LLM_SLUG}'

PANGEA_DIR_CANDIDATES = [
    BASE / 'data/experiment/pangea',
    BASE / 'data/experiment/pangaea_datasets',
]


def resolve_pangea_dir() -> Path:
    for path in PANGEA_DIR_CANDIDATES:
        if path.is_dir():
            return path
    searched = '\n  '.join(str(p) for p in PANGEA_DIR_CANDIDATES)
    raise FileNotFoundError(f'PANGAEA data directory not found. Tried:\n  {searched}')


def collect_pangea_csv_files(data_dir: Path) -> list[Path]:
    return sorted(data_dir.glob('*.csv'))


PANGEA_DIR = resolve_pangea_dir()
OUTPUT_DIR = BASE / 'outputs' / EXPERIMENT_RUN
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
for handler in logger.handlers[:]:
    logger.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
logger.addHandler(handler)

metadata_standard = METADATA_STANDARDS[STANDARD_NAME]


def output_path_for(csv_path: Path) -> Path:
    return OUTPUT_DIR / f'{csv_path.stem}_pangaea.json'


all_csv_files = collect_pangea_csv_files(PANGEA_DIR)
if not all_csv_files:
    raise FileNotFoundError(f'No CSV files found in {PANGEA_DIR}')

csv_files = [p for p in all_csv_files if not output_path_for(p).exists()]
skipped_count = len(all_csv_files) - len(csv_files)

print(f'LLM provider: {LLM_PROVIDER}')
print(f'LLM model: {LLM_NAME}')
print(f'Metadata standard: {STANDARD_NAME}')
print(f'Experiment run: {EXPERIMENT_RUN}')
print(f'Data directory: {PANGEA_DIR}')
print(f'Found {len(all_csv_files)} CSV file(s)')
print(f'Skipping {skipped_count} already completed in {OUTPUT_DIR}')
print(f'Remaining to run: {len(csv_files)}')
print(f'Output directory: {OUTPUT_DIR}')


LLM provider: surf
LLM model: default-text-large
Metadata standard: croissant_inaturalist_standard
Experiment run: pangaea_inat_default-text-large
Data directory: /home/com3dian/Github/metadata_agent/data/experiment/pangaea_datasets
Found 75 CSV file(s)
Skipping 0 already completed in /home/com3dian/Github/metadata_agent/outputs/pangaea_inat_default-text-large
Remaining to run: 75
Output directory: /home/com3dian/Github/metadata_agent/outputs/pangaea_inat_default-text-large


In [ ]:
SPATIAL_KEYS = ('min_lat', 'min_lon', 'max_lat', 'max_lon')
TEMPORAL_KEYS = ('from', 'to')


def normalize_inaturalist_metadata(metadata):
    """Keep only schema-shaped iNaturalist Croissant metadata."""
    if not isinstance(metadata, dict):
        return None
    if not {'spatialCoverage', 'temporalCoverage'}.issubset(metadata.keys()):
        return None

    spatial = metadata.get('spatialCoverage')
    temporal = metadata.get('temporalCoverage')

    if spatial is not None:
        if not isinstance(spatial, dict) or not set(SPATIAL_KEYS).issubset(spatial.keys()):
            return None
        spatial = {key: spatial[key] for key in SPATIAL_KEYS}

    if temporal is not None:
        if not isinstance(temporal, dict) or not set(TEMPORAL_KEYS).issubset(temporal.keys()):
            return None
        temporal = {key: temporal[key] for key in TEMPORAL_KEYS}

    if spatial is None and temporal is None:
        return None

    return {
        'spatialCoverage': spatial,
        'temporalCoverage': temporal,
    }


def metadata_from_step_results(metadata_result):
    """Fallback when synthesis leaves metadata_output empty."""
    for step in reversed(metadata_result.step_results or []):
        if step.player_role != 'croissant_inaturalist_metadata_generator':
            continue
        for player_result in step.individual_results or []:
            analysis = player_result.get('analysis')
            candidates = [analysis]
            if isinstance(analysis, str):
                candidates = [analysis.strip()]
                start = analysis.find('{')
                end = analysis.rfind('}')
                if start != -1 and end > start:
                    candidates.append(analysis[start:end + 1])
            for candidate in candidates:
                if isinstance(candidate, dict):
                    normalized = normalize_inaturalist_metadata(candidate)
                elif isinstance(candidate, str) and candidate:
                    try:
                        normalized = normalize_inaturalist_metadata(json.loads(candidate))
                    except json.JSONDecodeError:
                        normalized = None
                else:
                    normalized = None
                if normalized:
                    return normalized
    return None


def extract_metadata(metadata_result):
    if metadata_result is None:
        return None

    candidates = []
    if metadata_result.final_metadata:
        candidates.append(metadata_result.final_metadata)

    workspace = metadata_result.final_workspace or {}
    if workspace.get('metadata_output') is not None:
        candidates.append(workspace.get('metadata_output'))

    for candidate in candidates:
        normalized = normalize_inaturalist_metadata(candidate)
        if normalized:
            return normalized

    return metadata_from_step_results(metadata_result)


def is_empty_metadata(metadata):
    return normalize_inaturalist_metadata(metadata) is None


def run_pipeline(csv_path: Path, attempt: int = 1):
    context_name = f'pangea_{csv_path.stem}'
    print(f'\n{"=" * 60}')
    print(f'[{attempt}] Processing {csv_path.name} (context: {context_name})')

    return run_metadata_extraction(
        source=str(csv_path.resolve()),
        metadata_standard=metadata_standard,
        metadata_standard_name=STANDARD_NAME,
        name=context_name,
        topology_name=TOPOLOGY,
    )


results_summary = []
success_count = 0

for csv_path in csv_files:
    entry = {
        'csv': csv_path.name,
        'success': False,
        'output_file': None,
        'attempts': 0,
        'error': None,
    }

    metadata = None
    for attempt in (1, 2):
        entry['attempts'] = attempt
        try:
            result = run_pipeline(csv_path, attempt=attempt)
            metadata = extract_metadata(result)
        except Exception as exc:
            entry['error'] = str(exc)
            print(f'  Attempt {attempt} failed: {exc}')
            continue

        if not is_empty_metadata(metadata):
            break

        print(f'  Attempt {attempt} returned empty metadata' + (' — retrying' if attempt == 1 else ''))

    if is_empty_metadata(metadata):
        print(f'  FAILED: no metadata for {csv_path.name}')
        results_summary.append(entry)
        continue

    output_file = output_path_for(csv_path)
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    entry['success'] = True
    entry['output_file'] = str(output_file)
    success_count += 1
    print(f'  Saved: {output_file}')
    results_summary.append(entry)

print(f'\n{"=" * 60}')
print(f'Batch complete: {success_count}/{len(csv_files)} successful this run')
print(f'Previously completed (skipped): {skipped_count}')
print(f'Total with outputs: {skipped_count + success_count}/{len(all_csv_files)}')
for item in results_summary:
    status = 'OK' if item['success'] else 'FAIL'
    detail = item['output_file'] or item.get('error') or 'empty metadata'
    print(f'  [{status}] {item["csv"]} ({item["attempts"]} attempt(s)) -> {detail}')



[1] Processing 897882.csv (context: pangea_897882)


/home/com3dian/Github/metadata_agent/src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-16 09:15:00,843 - INFO - root - PlanExecutor initialized with topology: single
2026-07-16 09:15:00,844 - INFO - root -   Players per step: 1
2026-07-16 09:15:00,845 - INFO - root -   Debate rounds: 0
2026-07-16 09:15:00,845 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-07-16 09:15:00,845 - INFO - root - Orchestrator initialized with topology: single
2026-07-16 09:15:00,845 - INFO - root - ============================================================
2026-07-16 09:15:00,846 - INFO - root - STARTING ORCHESTRATION
2026-07-16 09:15:00,846 - INFO - root - Context: pangea_897882
2026-07-16 09:15:00,847 - INFO - root - Type: single_csv
2026-07-16 09:15:00,848 - INFO - root - Resources: ['897882']
2026-07-16 09:15:00,848 - INFO - root - Metadata standard: croissant_inaturalist_standard (structured output)
2026-07-16 09:15:00,849 - INFO - root - ======

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-16 09:15:15,982 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 09:15:15,988 - WARNING - root - Plan validation warning: Step 1 ('generate_inaturalist_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-16 09:15:15,988 - INFO - root - Plan generated successfully!
2026-07-16 09:15:15,988 - INFO - root - Number of steps: 2
2026-07-16 09:15:15,989 - INFO - root -   Step 1: detect_and_extract_spatial_temporal (player: spatial_temporal_specialist)
2026-07-16 09:15:15,989 - INFO - root -   Step 2: generate_inaturalist_metadata (player: croissant_inaturalist_metadata_generator)
2026-07-16 09:15:15,989 - INFO - root - Auto-added 'metadata_generator' for final metadata generation
2026-07-16 09:15:15,989 - INFO - root - Auto-added 'croissant_inaturalist_metadata_generator' for final metadata generation
2026-07-16 09:15:15,989 - INFO - root - Auto-added 'croissant_pangaea_metadata_generator' fo

...

2026-07-16 10:32:04,861 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 10:32:04,863 - INFO - root -   Player 'croissant_inaturalist_metadata_generator_1' completed execution
2026-07-16 10:32:04,864 - INFO - root -     Output: {"spatialCoverage":{"min_lat":11.93056667,"min_lon":-117.0224333,"max_lat":11.93056667,"max_lon":-117.0224333},"temporalCoverage":{"from":"2019-03-10T10:48:26+00:00","to":"2019-03-10T10:48:26+00:00"}}
2026-07-16 10:32:04,865 - INFO - root - Single player, skipping debate
2026-07-16 10:32:04,866 - INFO - root - --- STEP 1: SYNTHESIS ---
2026-07-16 10:32:04,867 - INFO - root -   Using structured output with schema: CroissantINaturalistMetadata
2026-07-16 10:32:08,757 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 10:32:16,333 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 10

/home/com3dian/Github/metadata_agent/src/orchestrator/orchestrator.py:400: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  orchestrator = Orchestrator(topology_name=topology_name)


2026-07-16 10:32:36,842 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 10:32:36,844 - WARNING - root - Plan validation warning: Step 1 ('generate_inaturalist_metadata') has unmet dependencies. Missing artifacts: {'metadata_standard'}
2026-07-16 10:32:36,844 - INFO - root - Plan generated successfully!
2026-07-16 10:32:36,845 - INFO - root - Number of steps: 2
2026-07-16 10:32:36,845 - INFO - root -   Step 1: detect_and_extract_spatial_temporal (player: spatial_temporal_specialist)
2026-07-16 10:32:36,846 - INFO - root -   Step 2: generate_inaturalist_metadata (player: croissant_inaturalist_metadata_generator)
2026-07-16 10:32:36,846 - INFO - root - Auto-added 'metadata_generator' for final metadata generation
2026-07-16 10:32:36,847 - INFO - root - Auto-added 'croissant_inaturalist_metadata_generator' for final metadata generation
2026-07-16 10:32:36,847 - INFO - root - Auto-added 'croissant_pangaea_metadata_generator' fo

/home/com3dian/Github/metadata_agent/src/players/player.py:971: UserWarning: Parameters {'extra_body'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  return Player(


2026-07-16 10:32:39,590 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-07-16 10:32:39,601 - INFO - root - Player 'spatial_temporal_specialist_1': LLM tool loop iteration 1 — model requested 1 tool(s): detect_temporal_columns
